In [0]:
# Data Quality Checks
# We check if the DATA is correct — not just if code ran

print("=" * 50)
print("STARTING DATA QUALITY CHECKS")
print("=" * 50)

# Load all bronze tables
bronze_orders = spark.table("bronze_orders")
bronze_customers = spark.table("bronze_customers")
bronze_order_items = spark.table("bronze_order_items")
bronze_payments = spark.table("bronze_payments")
bronze_products = spark.table("bronze_products")

# Track all failures
failures = []

print("\n--- BRONZE LAYER CHECKS ---")

# Check 1 — Row counts must be above 0
tables = {
    "bronze_orders": bronze_orders,
    "bronze_customers": bronze_customers,
    "bronze_order_items": bronze_order_items,
    "bronze_payments": bronze_payments,
    "bronze_products": bronze_products
}

for table_name, df in tables.items():
    row_count = df.count()
    if row_count == 0:
        failures.append(f"FAILED: {table_name} has 0 rows!")
        print(f"❌ {table_name}: 0 rows — FAILED!")
    else:
        print(f"✅ {table_name}: {row_count} rows — PASSED!")

print("\nBronze row count checks done!")

STARTING DATA QUALITY CHECKS

--- BRONZE LAYER CHECKS ---
✅ bronze_orders: 99441 rows — PASSED!
✅ bronze_customers: 99441 rows — PASSED!
✅ bronze_order_items: 112650 rows — PASSED!
✅ bronze_payments: 103886 rows — PASSED!
✅ bronze_products: 32951 rows — PASSED!

Bronze row count checks done!


In [0]:
# Check 2 — Critical columns must not have nulls
# order_id is the most important column — it must never be null

from pyspark.sql.functions import col

print("\n--- NULL CHECKS ON CRITICAL COLUMNS ---")

# Define which columns must never be null for each table
critical_columns = {
    "bronze_orders": ["order_id", "customer_id", "order_status"],
    "bronze_customers": ["customer_id", "customer_state"],
    "bronze_order_items": ["order_id", "product_id", "price"],
    "bronze_payments": ["order_id", "payment_value"],
    "bronze_products": ["product_id"]
}

for table_name, columns in critical_columns.items():
    df = spark.table(table_name)
    for column in columns:
        null_count = df.filter(col(column).isNull()).count()
        if null_count > 0:
            failures.append(
                f"FAILED: {table_name}.{column} has {null_count} nulls!"
            )
            print(f"❌ {table_name}.{column}: {null_count} nulls — FAILED!")
        else:
            print(f"✅ {table_name}.{column}: 0 nulls — PASSED!")

print("\nNull checks done!")


--- NULL CHECKS ON CRITICAL COLUMNS ---
✅ bronze_orders.order_id: 0 nulls — PASSED!
✅ bronze_orders.customer_id: 0 nulls — PASSED!
✅ bronze_orders.order_status: 0 nulls — PASSED!
✅ bronze_customers.customer_id: 0 nulls — PASSED!
✅ bronze_customers.customer_state: 0 nulls — PASSED!
✅ bronze_order_items.order_id: 0 nulls — PASSED!
✅ bronze_order_items.product_id: 0 nulls — PASSED!
✅ bronze_order_items.price: 0 nulls — PASSED!
✅ bronze_payments.order_id: 0 nulls — PASSED!
✅ bronze_payments.payment_value: 0 nulls — PASSED!
✅ bronze_products.product_id: 0 nulls — PASSED!

Null checks done!


In [0]:
# Check 3 — No duplicate order IDs allowed
# Every order must be unique — like a unique receipt number

print("\n--- DUPLICATE CHECKS ---")

# Check duplicate order_ids in bronze_orders
total_orders = bronze_orders.count()
unique_orders = bronze_orders.select("order_id").distinct().count()
duplicate_count = total_orders - unique_orders

if duplicate_count > 0:
    failures.append(
        f"FAILED: bronze_orders has {duplicate_count} duplicate order_ids!"
    )
    print(f"❌ bronze_orders: {duplicate_count} duplicates — FAILED!")
else:
    print(f"✅ bronze_orders: No duplicates — PASSED!")

# Check duplicate customer_ids in bronze_customers
total_customers = bronze_customers.count()
unique_customers = bronze_customers.select("customer_id").distinct().count()
duplicate_customers = total_customers - unique_customers

if duplicate_customers > 0:
    failures.append(
        f"FAILED: bronze_customers has {duplicate_customers} duplicate customer_ids!"
    )
    print(f"❌ bronze_customers: {duplicate_customers} duplicates — FAILED!")
else:
    print(f"✅ bronze_customers: No duplicates — PASSED!")

print("\nDuplicate checks done!")


--- DUPLICATE CHECKS ---
✅ bronze_orders: No duplicates — PASSED!
✅ bronze_customers: No duplicates — PASSED!

Duplicate checks done!


In [0]:
# Check 4 — Silver layer data quality checks
# After cleaning, silver must meet higher quality standards

print("\n--- SILVER LAYER CHECKS ---")

silver = spark.table("silver_orders_master")

# Check 1 — Silver must have more than 50000 rows
silver_count = silver.count()
if silver_count < 50000:
    failures.append(
        f"FAILED: silver_orders_master has only {silver_count} rows — too low!"
    )
    print(f"❌ Silver row count: {silver_count} — FAILED!")
else:
    print(f"✅ Silver row count: {silver_count} — PASSED!")

# Check 2 — Payment value must always be positive
from pyspark.sql.functions import col
negative_payments = silver.filter(col("payment_value") <= 0).count()
if negative_payments > 0:
    failures.append(
        f"FAILED: {negative_payments} rows have negative payment values!"
    )
    print(f"❌ Negative payments: {negative_payments} — FAILED!")
else:
    print(f"✅ No negative payments — PASSED!")

# Check 3 — No null customer states
null_states = silver.filter(col("customer_state").isNull()).count()
if null_states > 0:
    failures.append(
        f"FAILED: {null_states} rows have null customer_state!"
    )
    print(f"❌ Null customer states: {null_states} — FAILED!")
else:
    print(f"✅ No null customer states — PASSED!")

# Check 4 — All orders must have a payment
null_payments = silver.filter(col("payment_value").isNull()).count()
if null_payments > 0:
    failures.append(
        f"FAILED: {null_payments} orders have no payment value!"
    )
    print(f"❌ Null payment values: {null_payments} — FAILED!")
else:
    print(f"✅ No null payment values — PASSED!")

print("\nSilver checks done!")


--- SILVER LAYER CHECKS ---
✅ Silver row count: 115034 — PASSED!
✅ No negative payments — PASSED!
✅ No null customer states — PASSED!
❌ Null payment values: 3 — FAILED!

Silver checks done!


In [0]:
# Check 5 — Gold layer checks

print("\n--- GOLD LAYER CHECKS ---")

# Monthly revenue must have rows
gold_revenue = spark.table("gold_monthly_revenue")
if gold_revenue.count() == 0:
    failures.append("FAILED: gold_monthly_revenue is empty!")
    print("❌ gold_monthly_revenue: empty — FAILED!")
else:
    print(f"✅ gold_monthly_revenue: {gold_revenue.count()} rows — PASSED!")

# Top categories must have 10 rows
gold_categories = spark.table("gold_top_categories")
if gold_categories.count() < 10:
    failures.append("FAILED: gold_top_categories has less than 10 rows!")
    print("❌ gold_top_categories: less than 10 rows — FAILED!")
else:
    print(f"✅ gold_top_categories: {gold_categories.count()} rows — PASSED!")

print("\n" + "=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

if len(failures) == 0:
    print("✅ ALL CHECKS PASSED!")
    print("Data is clean and ready for dashboard!")
else:
    print(f"❌ {len(failures)} CHECKS FAILED:")
    for failure in failures:
        print(f"  → {failure}")
    raise Exception("Data quality checks failed! Pipeline stopped!")


--- GOLD LAYER CHECKS ---
✅ gold_monthly_revenue: 23 rows — PASSED!
✅ gold_top_categories: 10 rows — PASSED!

DATA QUALITY REPORT
❌ 1 CHECKS FAILED:
  → FAILED: 3 orders have no payment value!


---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-4829465103068622>, line 32
     30 for failure in failures:
     31     print(f"  → {failure}")
---> 32 raise Exception("Data quality checks failed! Pipeline stopped!")

Exception: Data quality checks failed! Pipeline stopped!